
# Cleaning Dataset 

Perform data cleaning on the SYNCH Rapid City Dataset, including formating the variable names, spell checking. 



In [2]:
%load_ext autoreload
%autoreload 2

# Import library

In [8]:

from sentence_transformers import SentenceTransformer, util , SimilarityFunction
from pathlib import Path


import pandas as pd
import numpy as np
import regex as re
import itertools as it
import matplotlib.pyplot as plt
import pickle
import os
import torch
import seaborn as sns 
import sys
import json
import networkx as nx

In [9]:

# Get the current working directory
cwd = Path.cwd()

# Get the path to the parent directory
parent_dir = cwd.parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

# Now you can import the data_io module from src
import src.data_io as data_io
import src.cld_nx as cld_nx

# Read data


In [10]:
print(os.getcwd())

/Users/gl665/PycharmProjects/Phrame_MentalMaps/notebook


## Read Raw data

In [11]:
# Go one level up from the current directory
# cwd = os.getcwd()

# parent_dir = Path(cwd).parent
# print(parent_dir)

# my_data_RC = {}
# folder_path = str(parent_dir / 'results/SYNCH/JSON_file/RC/')


# cld_data = data_io.read_json(folder_path, my_data_RC)

## Read clean data


In [12]:
cwd = os.getcwd()

parent_dir = Path(cwd).parent
print(parent_dir)

my_data = {}
folder_path = str(parent_dir / 'results/SYNCH/JSON_file/RC/clean')


cld_data = data_io.read_json(folder_path, my_data)

/Users/gl665/PycharmProjects/Phrame_MentalMaps
load: Passages Clean_edited.json
load: Wavi combined clean_edited.json


## Read already preprocessed data

In [ ]:
# dict_cld = {name_file: nxgraph}
def get_cld_graph(dict_cld, cld_name = str):
    cld_list = dict_cld[cld_name]
    G = nx.from_dict_of_dicts(cld_list[1], create_using=nx.DiGraph)

    G = nx.relabel_nodes(G, cld_list[0])

    return cld_list, G
dict_cld = {}

for k, v in my_data.items():
    cld_list, G_cld = get_cld_graph(my_data, k)
    # cld_name = dict_file_rename.get(k, "")
    dict_cld[k] = G_cld

list_of_G = list(dict_cld.values())

for k, v in dict_cld.items(): 
    print(k, ":", v)


In [7]:
# read the data 
# file_path = "results/SYNCH/JSON_file/RC_cld_data_cleaned_preprocessed.json"

# # Pickle dump the cld_data_new dictionary
# with open(file_path, "rb") as file:
#     cld_data = pickle.load(file)

# print(cld_data)



# Single CLD view

In [6]:
cld_data

{'Passages Clean_edited.json': [{1: 'accountablility',
   2: 'alcohol exposed pregnancy',
   3: 'alcohol misuse',
   4: 'belief in a higher power',
   5: 'coping skills',
   6: 'death',
   7: 'dire situtations',
   8: 'drug misuse',
   9: 'family emotional health',
   10: 'harmful romantic partners',
   11: 'healing',
   12: 'incarceration',
   13: 'intimate partner violence',
   14: 'losing family and children through justice or child welfare system involvement',
   15: 'mental health',
   16: 're-entry facility',
   17: 'resources that provide structure and support',
   18: 'supportive people'},
  {3: {13: {'weight': 1},
    9: {'weight': -1},
    7: {'weight': 1},
    15: {'weight': -1},
    12: {'weight': 3},
    2: {'weight': 1},
    8: {'weight': 1},
    14: {'weight': 1},
    18: {'weight': -1}},
   13: {3: {'weight': 1},
    10: {'weight': 1},
    7: {'weight': 3},
    15: {'weight': -1},
    12: {'weight': 1},
    6: {'weight': 1},
    8: {'weight': 1},
    14: {'weight': 1},


# save all variables to an excel sheet

In [ ]:
# create a excel sheet where the nodes are listed in column 1 and the cld that comes from is listed in column 2 

vars_G1 = list(list_of_G[0].nodes)

print(vars_G1)

df_vars = pd.DataFrame(vars_G1, columns = ['Variable'])
df_vars['CLD'] = "Passage"


df_vars.to_excel('var_names.xlsx', sheet_name='passage', index=False)



In [ ]:
try:
    writer = pd.ExcelWriter('output.xlsx', engine='xlsxwriter')
    df1.to_excel(writer, sheet_name='Sheet1', index=False)
    df2.to_excel(writer, sheet_name='Sheet2', index=False)
    df3.to_excel(writer, sheet_name='Sheet3', index=False)
    writer.close()
except Exception as e:
    print(f"An error occurred: {e}")

# Cleaning Vairable Names 

## Clean the format of the vairable names 

In [9]:

# clean variable format 
i = 0
variable_clean_dict = {}
for k, v in cld_data.items():
    for key, value in v[0].items():
        
        correct_value = cld_nx.clean_variable_name(value)
        if value != correct_value: 
            variable_clean_dict[value]= correct_value
            i += 1

        v[0][key] = correct_value    
        


print('---')
print("Number of variables that corrected due to formating issue :", i)


---
Number of variables that corrected due to formating issue : 0


In [9]:
# with open("2_clean_format_dictionary.txt", "w") as fp:
#     json.dump(variable_clean_dict, fp) 


In [20]:
# write dictionary to a excel sheet, 
# with the original variable name and the corrected variable name
df_1 = pd.DataFrame(variable_clean_dict.items(), columns=['Original', 'Corrected'])


## Spell Checking 

In [11]:
# TODO: Q: Need to refine these 

custom_words = ['american indian and alaska native', 
            'indian', 
            'alaska',
            'christianity', 
            'fostercare', 
            'american',
            'child-rearing',
            'reoffending', 
            'lakota',
            'mitakuye oyasin', 
            'pushback',
            'prosocial']


In [21]:

from spellchecker import SpellChecker

def correct_spelling(text, custom_words = []):
    spell = SpellChecker()
    spell.word_frequency.load_words(custom_words)
    
    words = text.split()
    corrected_words = [spell.correction(word) or word for word in words]
    return " ".join(corrected_words)


In [23]:
i = 0
variable_correct_spell = {}
for k, v in my_data.items():
    for key, value in v[0].items():
        corrected_text = correct_spelling(value, custom_words)
        if value != corrected_text: 
            variable_correct_spell[value] = corrected_text
            # print(f'Original: {value}')
            # print(f'Corrected: {corrected_text}')
            i += 1
        
        v[0][key] = corrected_text    
print('---')
print("Number of variables found needed spell check:", i)

---
Number of variables found needed spell check: 3


In [24]:
variable_correct_spell

{'accountablility': 'accountability',
 'dire situtations': 'dire situations',
 're-entry facility': 'reentry facility'}

In [8]:
# with open("3_spell_check_dictionary.txt", "w") as fp:
#     json.dump(variable_correct_spell, fp) 

In [25]:
df_2 = pd.DataFrame(variable_correct_spell.items(), columns=['Original', 'Corrected'])

In [ ]:

# with pd.ExcelWriter('results/SYNCH/file_rename/RC_variables_matching060424.xlsx') as writer:
# # Adding dataframes to Excel as new sheets
#     df_1.to_excel(writer, sheet_name='1.clean format', index=False)
#     df_2.to_excel(writer, sheet_name='2.spellcheck', index=False)


# Find semantic similary variable names using pretrained sentence transformer

In [ ]:
# variable_clean_dict = {}
# for k, v in cld_data.items():
#     for key, value in v[0].items():
        
#         correct_value = clean_variable_name(value)
#         if value != correct_value: 
#             variable_clean_dict[value]= correct_value
#             i += 1

#         v[0][key] = correct_value    
        

In [26]:
variable_names = []

for k, v in cld_data.items():
    cld_new_vars_name = []
    cld_var_names = list(v[0].values())

    variable_names.extend(cld_var_names)


print("number of originial nodes: ", len(variable_names))

variable_names = np.sort(list(set(variable_names)))
print("number of originial unique nodes: ", len(variable_names))


number of originial nodes:  53
number of originial unique nodes:  49


In [27]:
# print the first 10 words of variable names
print(variable_names[:10])

['accountability' 'adverse childhood experiences'
 'alcohol exposed pregnancy' 'alcohol misuse' 'approval of therapy'
 'being self sufficient' 'belief in a higher power'
 'child being abused in home' 'child welfare system intervention'
 'children removed from house']


In [28]:
# Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-mpnet-base-v2")

embeddings = model.encode(variable_names)

similarities = model.similarity(embeddings, embeddings)



/Users/gl665/PycharmProjects/Phrame_MentalMaps/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# Output the pairs with their score
# for idx_i, sentence1 in enumerate(variable_names):
#     print(sentence1)
#     for idx_j, sentence2 in enumerate(variable_names):
#         if similarities[idx_i][idx_j] > 0.75:
#             print(f" - {sentence2: <30}: {similarities[idx_i][idx_j]:.4f}")
#             break 

In [29]:
#TODO: see the fine tuning notebook
# TODO: set the threshold based on differnt perfromance 
# Threshold for considering names as similar
threshold = 0.75

# Group semantically similar names
groups = []
visited = set()

for idx, name in enumerate(variable_names):
    if name not in visited:
        similar_names = [name]
        for jdx, similarity in enumerate(similarities[idx]):
            if jdx != idx and similarity >= threshold and variable_names[jdx] not in visited:
                similar_names.append(variable_names[jdx])
                visited.add(variable_names[jdx])
                
        groups.append(similar_names)
        visited.update(similar_names)


In [30]:
print("numbers of final variables after grouping: ", len(groups))


numbers of final variables after grouping:  46


In [31]:
# save groups to excel sheet
df_merge_groups = pd.DataFrame(groups)
# df_merge_groups.to_excel('results/SYNCH/file_rename/RC_variables_grouped060424.xlsx', index=False)

In [32]:

# Print grouped variable names

for i, group in enumerate(groups):
    if len(group) > 1:
        print(f"Group {i + 1}: {group}")

len(groups)

Group 13: ['community acceptance of intimate partner violence', 'community intimate partner violence rate', 'family and friend permissive intimate partner violence norms']
Group 26: ['healthy relationship functioning', 'knowledge about healthy relationships']


46

In [33]:
# create a dictionary to swap the words that are bundled in the group. 
swap_dict = {}

for group in groups:
    if len(group) > 1:
        longest_name = max(group, key=len)
        swap_dict[longest_name] = group
swap_dict

swap_dict_reversed = {v: k for k, vv in swap_dict.items() for v in vv}
swap_dict_reversed


{'community acceptance of intimate partner violence': 'family and friend permissive intimate partner violence norms',
 'community intimate partner violence rate': 'family and friend permissive intimate partner violence norms',
 'family and friend permissive intimate partner violence norms': 'family and friend permissive intimate partner violence norms',
 'healthy relationship functioning': 'knowledge about healthy relationships',
 'knowledge about healthy relationships': 'knowledge about healthy relationships'}

In [34]:
len(swap_dict_reversed)

5

In [35]:
# Swap the variable names in the data

cld_data_new = cld_data.copy()

for k, v in cld_data_new.items():
    for key, value in v[0].items():
        if value in swap_dict_reversed:
            v[0][key] = swap_dict_reversed[value]



In [36]:
variable_names = [list(v[0].values()) for k, v in cld_data_new.items()]

variable_names = sum(variable_names, [])

variable_names = np.sort(list(set(variable_names)))


print("number of originial unique nodes after cleaning the data: ", len(variable_names))


number of originial unique nodes after cleaning the data:  46


# write out the cleaned data to pickle

In [ ]:
# # Specify the file path where you want to save the pickled data
file_path = "results/SYNCH/JSON_file/RC_cld_data_cleaned_and_merged.json"

merged_file_path = str(parent_dir / file_path)

# Pickle dump the cld_data_new dictionary
with open(merged_file_path, "wb") as file:
    pickle.dump(cld_data_new, file)


c:\Users\GL665\Documents\Phrame-MentalMaps\results\SYNCH\JSON_file\RC_cld_data_cleaned_and_merged.json
